# TALE Multi-output Sequence Regression Training Notebook

This notebook trains a PyTorch multi-output regression model from 44-46 nt DNA sequences to four continuous phenotypes. The default training table is `TALE_train_data_260312.csv`, distributed separately from this code repository rather than committed here because of its size.

## Outputs

The selected 2026-03-20 model uses seed 3407, EMA weights, and the run configuration summarized in `../model/summary.json`.

In [ ]:
# Install packages in a fresh environment if needed.
# pip install torch pandas numpy scipy scikit-learn matplotlib jupyter


In [ ]:
import os
import re
import sys
import json
import math
import time
import random
import shutil
import inspect
import platform
from pathlib import Path
from datetime import datetime
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [ ]:
# =========================
# Configuration note.
# =========================
def as_int(x, name, min_value=None, max_value=None):
    if isinstance(x, bool):
        raise TypeError(f"{name} must not be bool")
    if isinstance(x, (int, np.integer)):
        value = int(x)
    elif isinstance(x, float) and float(x).is_integer():
        value = int(x)
    else:
        raise TypeError(f"{name} must be an integer; value={x!r}, type={type(x)}")
    if min_value is not None and value < min_value:
        raise ValueError(f"{name} must be >= {min_value}, value={value}")
    if max_value is not None and value > max_value:
        raise ValueError(f"{name} must be <= {max_value}, value={value}")
    return value


def as_float(x, name, min_value=None, max_value=None):
    if isinstance(x, bool):
        raise TypeError(f"{name} must not be bool")
    value = float(x)
    if min_value is not None and value < min_value:
        raise ValueError(f"{name} must be >= {min_value}, value={value}")
    if max_value is not None and value > max_value:
        raise ValueError(f"{name} must be <= {max_value}, value={value}")
    return value


def to_builtin(x):
    if isinstance(x, dict):
        return {str(k): to_builtin(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_builtin(v) for v in x]
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if torch.is_tensor(x):
        return x.detach().cpu().tolist()
    return x


def write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_builtin(obj), f, ensure_ascii=False, indent=2)


def safe_slug(s):
    s = str(s).strip()
    s = re.sub(r"[^A-Za-z0-9._-]+", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s or "run"


def query_nvidia_smi():
    if shutil.which("nvidia-smi") is None:
        return None
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,name,temperature.gpu,utilization.gpu,memory.used,memory.total",
        "--format=csv,noheader,nounits"
    ]
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    except Exception as e:
        print(f"nvidia-smi query failed: {e}")
        return None

    rows = []
    for line in result.stdout.strip().splitlines():
        parts = [x.strip() for x in line.split(",")]
        if len(parts) != 6:
            continue
        idx, name, temp_c, util, mem_used, mem_total = parts
        rows.append({
            "gpu_index": int(idx),
            "name": name,
            "temp_C": float(temp_c),
            "util_%": float(util),
            "mem_used_MiB": float(mem_used),
            "mem_total_MiB": float(mem_total),
            "mem_free_MiB": float(mem_total) - float(mem_used),
        })
    if not rows:
        return None
    return pd.DataFrame(rows).sort_values("gpu_index").reset_index(drop=True)


def show_gpu_status():
    print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
    print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")
    df_gpu = query_nvidia_smi()
    if df_gpu is not None:
        display(df_gpu)
    else:
        print("Could not read GPU status with nvidia-smi.")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            prop = torch.cuda.get_device_properties(i)
            print(f"[torch] cuda:{i} | {prop.name} | total_memory={prop.total_memory / 1024**3:.2f} GiB")


def select_device(gpu_index=0):
    if not torch.cuda.is_available():
        print("CUDA is unavailable; using CPU.")
        return torch.device("cpu")
    gpu_index = as_int(gpu_index, "GPU_INDEX", min_value=0)
    n_gpu = torch.cuda.device_count()
    if gpu_index >= n_gpu:
        raise ValueError(f"GPU_INDEX={gpu_index} is out of range; visible GPU count = {n_gpu}")
    torch.cuda.set_device(gpu_index)
    return torch.device(f"cuda:{gpu_index}")


def get_device_summary(device):
    if device.type != "cuda":
        return {"device": str(device), "name": "CPU"}
    idx = device.index if device.index is not None else torch.cuda.current_device()
    prop = torch.cuda.get_device_properties(idx)
    return {
        "device": f"cuda:{idx}",
        "name": prop.name,
        "total_memory_GiB": round(prop.total_memory / 1024**3, 3),
    }


def save_notebook_snapshot(src_path, dst_path):
    src_path = Path(src_path)
    dst_path = Path(dst_path)
    if src_path.exists():
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, dst_path)
        return True
    return False


def build_run_name(base_name, tag="", timestamp=None):
    timestamp = timestamp or datetime.now().strftime("%y%m%d_%H%M%S")
    parts = [safe_slug(base_name)]
    if tag:
        parts.append(safe_slug(tag))
    parts.append(timestamp)
    return "_".join(parts)


def maybe_float(v, default=np.nan):
    try:
        return float(v)
    except Exception:
        return default


print("Status message.")

In [ ]:
# =========================
# Configuration note.
# =========================

# -------------------------
# 1) Data
# -------------------------
CSV_PATH = "TALE_train_data_260312.csv"
SEQ_COL = "Nn"
GROUP_COL = "group"
TARGET_COLS = [
    "cyt.score.A549",
    "nuc.score.A549",
    "cyt.score.HCT116",
    "nuc.score.HCT116",
]

# Configuration note.
EVALUATE_INDEPENDENT_A549 = True
INDEPENDENT_A549_CSV_PATH = "3pL6-A549-T1.csv"
INDEPENDENT_A549_TARGET_COLS = ["cyt.score", "nuc.score"]
INDEPENDENT_A549_MODEL_TARGET_COLS = ["cyt.score.A549", "nuc.score.A549"]

MAX_LEN = as_int(46, "MAX_LEN", min_value=1)

# -------------------------
# 2) Experiment naming and outputs
# -------------------------
EXPERIMENT_NAME = "TALE"
EXPERIMENT_TAG = "e5-lstm256bx128b-fc256*0.1-ema0.999-seed3407"

SAVE_DEBUG_ARTIFACTS = True
SAVE_PREDICTIONS = True
SAVE_SUMMARY_JSON = True

# -------------------------
# 3) Device and training loop
# -------------------------
GPU_INDEX = as_int(7, "GPU_INDEX", min_value=0)
SEED = as_int(3407, "SEED", min_value=0)

BATCH_SIZE = as_int(2048, "BATCH_SIZE", min_value=1)
NUM_WORKERS = as_int(4, "NUM_WORKERS", min_value=0)
PIN_MEMORY = True
PERSISTENT_WORKERS = True

NUM_EPOCHS = as_int(128, "NUM_EPOCHS", min_value=1)
MAX_TRAIN_STEPS = as_int(0, "MAX_TRAIN_STEPS", min_value=0)
EVAL_EVERY_N_STEPS = as_int(50, "EVAL_EVERY_N_STEPS", min_value=1)
GRAD_CLIP_NORM = as_float(1.0, "GRAD_CLIP_NORM", min_value=0.0)

# -------------------------
# 4) Model:embedding / CNN / LSTM / readout / FC
# -------------------------
EMBED_DIM = as_int(5, "EMBED_DIM", min_value=1)

USE_POSITIONAL_EMBEDDING = False
POSITIONAL_DROPOUT = as_float(0.0, "POSITIONAL_DROPOUT", min_value=0.0, max_value=1.0)

# Configuration note.
USE_CNN_BEFORE_LSTM = False
CNN_OUT_CHANNELS = [64]
CNN_KERNEL_SIZES = [9]
CNN_STRIDES = [1]
CNN_POOL_TYPES = ["none"]
CNN_POOL_SIZES = [1]
CNN_ACTIVATION = "relu"
CNN_DROPOUT = as_float(0.0, "CNN_DROPOUT", min_value=0.0, max_value=1.0)

# Configuration note.
LSTM_HIDDEN_DIMS = [256, 128]
LSTM_BIDIRECTIONAL = [True, True]
USE_PACKED_LSTM = False

LSTM_INTER_LAYER_DROPOUT = as_float(0.0, "LSTM_INTER_LAYER_DROPOUT", min_value=0.0, max_value=1.0)
LSTM_OUTPUT_DROPOUT = as_float(0.0, "LSTM_OUTPUT_DROPOUT", min_value=0.0, max_value=1.0)

READOUT_MODE = "flatten"       # "flatten" / "attention"
ATTENTION_HIDDEN_DIM = as_int(64, "ATTENTION_HIDDEN_DIM", min_value=1)

FC_HIDDEN_DIMS = [256]
FC_INTER_LAYER_DROPOUT = as_float(0.0, "FC_INTER_LAYER_DROPOUT", min_value=0.0, max_value=1.0)
FC_OUTPUT_DROPOUT = as_float(0.1, "FC_OUTPUT_DROPOUT", min_value=0.0, max_value=1.0)

# -------------------------
# Configuration note.
# -------------------------
OPTIMIZER_NAME = "AdamW"       # "Adam" / "AdamW"
LR = as_float(7e-4, "LR", min_value=0.0)
WEIGHT_DECAY = as_float(1e-5, "WEIGHT_DECAY", min_value=0.0)

# EMA(Exponential Moving Average)
# Configuration note.
USE_EMA = True
EMA_DECAY = as_float(0.999, "EMA_DECAY", min_value=0.0, max_value=1.0)
EMA_UPDATE_AFTER_STEP = as_int(0, "EMA_UPDATE_AFTER_STEP", min_value=0)
EMA_UPDATE_EVERY = as_int(1, "EMA_UPDATE_EVERY", min_value=1)
EMA_EVAL_WITH_EMA_WEIGHTS = True

USE_WARMUP = False
WARMUP_STEPS = as_int(128, "WARMUP_STEPS", min_value=0)
WARMUP_START_FACTOR = as_float(0.01, "WARMUP_START_FACTOR", min_value=0.0, max_value=1.0)

USE_SCHEDULER = False
SCHEDULER_FACTOR = as_float(0.5, "SCHEDULER_FACTOR", min_value=0.0, max_value=1.0)
SCHEDULER_PATIENCE = as_int(2, "SCHEDULER_PATIENCE", min_value=1)

EARLY_STOPPING_PATIENCE = as_int(16, "EARLY_STOPPING_PATIENCE", min_value=1)
EARLY_STOPPING_MIN_DELTA = as_float(0.0, "EARLY_STOPPING_MIN_DELTA", min_value=0.0)


In [ ]:
# =========================
# Configuration note.
# =========================

# Configuration note.
VOCAB = {"A": 0, "C": 1, "G": 2, "T": 3, "N": 4, "PAD": 5}
PAD_TOKEN = "PAD"
PAD_ID = as_int(VOCAB[PAD_TOKEN], "PAD_ID", min_value=0)
N_ID = as_int(VOCAB["N"], "N_ID", min_value=0)
ID2BASE = {v: k for k, v in VOCAB.items()}

RUNS_ROOT = Path("runs")
CURRENT_NOTEBOOK_PATH = Path("TALE_training.ipynb")
COPY_NOTEBOOK_SNAPSHOT = bool(SAVE_DEBUG_ARTIFACTS)
USE_EMA = bool(USE_EMA)
EMA_EVAL_WITH_EMA_WEIGHTS = bool(EMA_EVAL_WITH_EMA_WEIGHTS and USE_EMA)

# Configuration note.
LSTM_HIDDEN_DIMS = [as_int(x, f"LSTM_HIDDEN_DIMS[{i}]", min_value=1) for i, x in enumerate(LSTM_HIDDEN_DIMS)]
LSTM_BIDIRECTIONAL = [bool(x) for x in LSTM_BIDIRECTIONAL]
FC_HIDDEN_DIMS = [as_int(x, f"FC_HIDDEN_DIMS[{i}]", min_value=1) for i, x in enumerate(FC_HIDDEN_DIMS)]

CNN_OUT_CHANNELS = [as_int(x, f"CNN_OUT_CHANNELS[{i}]", min_value=1) for i, x in enumerate(CNN_OUT_CHANNELS)]
CNN_KERNEL_SIZES = [as_int(x, f"CNN_KERNEL_SIZES[{i}]", min_value=1) for i, x in enumerate(CNN_KERNEL_SIZES)]
CNN_STRIDES = [as_int(x, f"CNN_STRIDES[{i}]", min_value=1) for i, x in enumerate(CNN_STRIDES)]
CNN_POOL_TYPES = [str(x).lower() for x in CNN_POOL_TYPES]
CNN_POOL_SIZES = [as_int(x, f"CNN_POOL_SIZES[{i}]", min_value=1) for i, x in enumerate(CNN_POOL_SIZES)]
CNN_ACTIVATION = str(CNN_ACTIVATION).lower()

if len(LSTM_HIDDEN_DIMS) == 0:
    raise ValueError("LSTM_HIDDEN_DIMS must not be empty; this model requires at least one LSTM layer.")
if len(LSTM_HIDDEN_DIMS) != len(LSTM_BIDIRECTIONAL):
    raise ValueError("LSTM_HIDDEN_DIMS and LSTM_BIDIRECTIONAL must have the same length.")

if READOUT_MODE not in {"flatten", "attention"}:
    raise ValueError("READOUT_MODE must be 'flatten' or 'attention'.")

if CNN_ACTIVATION not in {"relu"}:
    raise ValueError("CNN_ACTIVATION currently supports only 'relu'.")

cnn_lengths = [len(CNN_OUT_CHANNELS), len(CNN_KERNEL_SIZES), len(CNN_STRIDES), len(CNN_POOL_TYPES), len(CNN_POOL_SIZES)]
if len(set(cnn_lengths)) != 1:
    raise ValueError("CNN parameter lists must have the same length.")
for i, pool_type in enumerate(CNN_POOL_TYPES):
    if pool_type not in {"none", "max"}:
        raise ValueError(f"CNN_POOL_TYPES[{i}] must be 'none' or 'max'.")

show_gpu_status()
DEVICE = select_device(GPU_INDEX)
PIN_MEMORY = bool(PIN_MEMORY and DEVICE.type == "cuda")
PERSISTENT_WORKERS = bool(PERSISTENT_WORKERS and NUM_WORKERS > 0)

RUN_NAME = build_run_name(EXPERIMENT_NAME, EXPERIMENT_TAG)
RUN_DIR = RUNS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_TMP_PATH = RUN_DIR / "_best_checkpoint_tmp.pt"
FINAL_MODEL_SAVE_PATH = RUN_DIR / "model_bundle.pt"
SUMMARY_PATH = RUN_DIR / "summary.json"
OUT_TEST_PRED = RUN_DIR / "test_predictions.csv"
OUT_INDEPENDENT_A549_PRED = RUN_DIR / "independent_a549_predictions.csv"

# Configuration note.
STEP_EVAL_HISTORY_PATH = RUN_DIR / "history_step_eval.csv"
EPOCH_HISTORY_PATH = RUN_DIR / "history_epoch.csv"
TRAIN_CURVE_PATH = RUN_DIR / "training_curves_step_eval.png"
TEST_SCATTER_PATH = RUN_DIR / "test_prediction_scatter.png"
INDEPENDENT_A549_SCATTER_PATH = RUN_DIR / "independent_a549_prediction_scatter.png"
NOTEBOOK_SNAPSHOT_PATH = RUN_DIR / CURRENT_NOTEBOOK_PATH.name

print(f"Using device: {DEVICE}")
print(f"RUN_DIR: {RUN_DIR}")

display(pd.DataFrame([{
    "BATCH_SIZE": BATCH_SIZE,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "NUM_EPOCHS": NUM_EPOCHS,
    "NUM_WORKERS": NUM_WORKERS,
    "PIN_MEMORY": PIN_MEMORY,
    "PERSISTENT_WORKERS": PERSISTENT_WORKERS,
    "OPTIMIZER_NAME": OPTIMIZER_NAME,
    "EXPERIMENT_TAG": EXPERIMENT_TAG,
    "USE_EMA": USE_EMA,
    "EMA_DECAY": EMA_DECAY if USE_EMA else None,
    "EMA_UPDATE_AFTER_STEP": EMA_UPDATE_AFTER_STEP if USE_EMA else None,
    "EMA_UPDATE_EVERY": EMA_UPDATE_EVERY if USE_EMA else None,
    "EMA_EVAL_WITH_EMA_WEIGHTS": EMA_EVAL_WITH_EMA_WEIGHTS if USE_EMA else None,
    "USE_CNN_BEFORE_LSTM": USE_CNN_BEFORE_LSTM,
    "CNN_LAYERS": len(CNN_OUT_CHANNELS),
    "CNN_OUT_CHANNELS": str(CNN_OUT_CHANNELS),
    "CNN_KERNEL_SIZES": str(CNN_KERNEL_SIZES),
    "CNN_STRIDES": str(CNN_STRIDES),
    "CNN_POOL_TYPES": str(CNN_POOL_TYPES),
    "CNN_POOL_SIZES": str(CNN_POOL_SIZES),
    "CNN_DROPOUT": CNN_DROPOUT,
    "LSTM_LAYERS": len(LSTM_HIDDEN_DIMS),
    "LSTM_HIDDEN_DIMS": str(LSTM_HIDDEN_DIMS),
    "LSTM_BIDIRECTIONAL": str(LSTM_BIDIRECTIONAL),
    "LSTM_INTER_LAYER_DROPOUT": LSTM_INTER_LAYER_DROPOUT,
    "LSTM_OUTPUT_DROPOUT": LSTM_OUTPUT_DROPOUT,
    "USE_POSITIONAL_EMBEDDING": USE_POSITIONAL_EMBEDDING,
    "USE_PACKED_LSTM": USE_PACKED_LSTM,
    "READOUT_MODE": READOUT_MODE,
    "ATTENTION_HIDDEN_DIM": ATTENTION_HIDDEN_DIM if READOUT_MODE == "attention" else None,
    "FC_LAYERS": len(FC_HIDDEN_DIMS),
    "FC_HIDDEN_DIMS": str(FC_HIDDEN_DIMS),
    "FC_INTER_LAYER_DROPOUT": FC_INTER_LAYER_DROPOUT,
    "FC_OUTPUT_DROPOUT": FC_OUTPUT_DROPOUT,
    "USE_WARMUP": USE_WARMUP,
    "WARMUP_STEPS": WARMUP_STEPS,
    "WARMUP_START_FACTOR": WARMUP_START_FACTOR,
    "EVAL_EVERY_N_STEPS": EVAL_EVERY_N_STEPS,
}]))


## Training Artifacts and Reproducibility

Each run writes an isolated directory under `runs/<RUN_NAME>/`, including model configuration, preprocessing configuration, training configuration, metadata, histories, curves, test predictions, and the final `model_bundle.pt`.

In [ ]:
# =========================
# Random seed
# =========================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Configuration note.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Seed fixed at {SEED}")

In [ ]:
# =========================
# Configuration note.
# =========================
df = pd.read_csv(CSV_PATH)
print(df.shape)
display(df.head())

required_cols = [SEQ_COL, GROUP_COL] + TARGET_COLS
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV is missing these columns: {missing}")

print(df[GROUP_COL].value_counts(dropna=False))


In [ ]:
# Basic checks
print("Sequence length summary:")
seq_lengths = df[SEQ_COL].astype(str).str.len()
display(seq_lengths.describe())

print("\nAny NA in required columns?")
display(df[required_cols].isna().sum())

# Configuration note.
before_n = len(df)
df = df.dropna(subset=required_cols).copy()
after_n = len(df)
print(f"Dropped {before_n - after_n} rows with NA in required columns.")

# Configuration note.
df[GROUP_COL] = df[GROUP_COL].astype(str).str.strip().str.lower()
valid_groups = {"train", "val", "test"}
bad_groups = sorted(set(df[GROUP_COL]) - valid_groups)
if bad_groups:
    raise ValueError(f"The group column contains invalid labels: {bad_groups}")

# Configuration note.
df[SEQ_COL] = df[SEQ_COL].astype(str).str.upper().str.strip()

display(df.head())


In [ ]:

# =========================
# Sequence tokenization
# Configuration note.
# - A/C/G/T -> 0/1/2/3
# Configuration note.
# - padding -> PAD token (= 5)
# Configuration note.
# =========================
def encode_sequence(seq: str, max_len: int = 46, vocab: dict = VOCAB, pad_id: int = PAD_ID):
    seq = str(seq).upper().strip()
    token_ids = []
    for ch in seq[:max_len]:
        token_ids.append(vocab.get(ch, vocab["N"]))

    seq_len = min(len(token_ids), max_len)

    if len(token_ids) < max_len:
        token_ids.extend([pad_id] * (max_len - len(token_ids)))

    return token_ids, seq_len

# Configuration note.
example_seq = df.iloc[0][SEQ_COL]
example_token_ids, example_len = encode_sequence(example_seq, MAX_LEN)
print(example_seq)
print(example_token_ids)
print("encoded length:", len(example_token_ids))
print("true seq length:", example_len)
print("decoded:", "".join(ID2BASE[x] for x in example_token_ids))


In [ ]:
# Configuration note.
train_df = df[df[GROUP_COL] == "train"].reset_index(drop=True).copy()
val_df   = df[df[GROUP_COL] == "val"].reset_index(drop=True).copy()
test_df  = df[df[GROUP_COL] == "test"].reset_index(drop=True).copy()

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
    raise ValueError("At least one of train / val / test is empty; check the group column.")


# Configuration note.
independent_a549_df = None
if EVALUATE_INDEPENDENT_A549:
    independent_a549_df = pd.read_csv(INDEPENDENT_A549_CSV_PATH)
    print("independent_a549:", independent_a549_df.shape)
    display(independent_a549_df.head())

    independent_required_cols = [SEQ_COL] + INDEPENDENT_A549_TARGET_COLS
    independent_missing = [c for c in independent_required_cols if c not in independent_a549_df.columns]
    if independent_missing:
        raise ValueError(f"Independent A549 test set is missing these columns: {independent_missing}")

    before_n_ind = len(independent_a549_df)
    independent_a549_df = independent_a549_df.dropna(subset=independent_required_cols).copy()
    after_n_ind = len(independent_a549_df)
    print(f"Dropped {before_n_ind - after_n_ind} rows with NA in independent A549 set.")

    independent_a549_df[SEQ_COL] = independent_a549_df[SEQ_COL].astype(str).str.upper().str.strip()
    display(independent_a549_df.head())


In [ ]:

# =========================
# Configuration note.
# =========================
train_y = train_df[TARGET_COLS].values.astype(np.float32)
target_mean = train_y.mean(axis=0)
target_std = train_y.std(axis=0)

# Configuration note.
target_std = np.where(target_std < 1e-8, 1.0, target_std)

target_stats = pd.DataFrame({
    "target": TARGET_COLS,
    "mean": target_mean,
    "std": target_std
})
display(target_stats)

split_summary = {
    "n_total": int(len(df)),
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
}

model_config = {
    "model_class": "TALELSTMRegressor",
    "vocab_size": int(len(VOCAB)),
    "pad_id": int(PAD_ID),
    "embed_dim": int(EMBED_DIM),
    "max_len": int(MAX_LEN),
    "use_positional_embedding": bool(USE_POSITIONAL_EMBEDDING),
    "positional_dropout": float(POSITIONAL_DROPOUT),
    "use_cnn_before_lstm": bool(USE_CNN_BEFORE_LSTM),
    "cnn_out_channels": [int(x) for x in CNN_OUT_CHANNELS],
    "cnn_kernel_sizes": [int(x) for x in CNN_KERNEL_SIZES],
    "cnn_strides": [int(x) for x in CNN_STRIDES],
    "cnn_pool_types": [str(x) for x in CNN_POOL_TYPES],
    "cnn_pool_sizes": [int(x) for x in CNN_POOL_SIZES],
    "cnn_activation": str(CNN_ACTIVATION),
    "cnn_dropout": float(CNN_DROPOUT),
    "lstm_hidden_dims": [int(x) for x in LSTM_HIDDEN_DIMS],
    "lstm_bidirectional": [bool(x) for x in LSTM_BIDIRECTIONAL],
    "lstm_inter_layer_dropout": float(LSTM_INTER_LAYER_DROPOUT),
    "lstm_output_dropout": float(LSTM_OUTPUT_DROPOUT),
    "use_packed_lstm": bool(USE_PACKED_LSTM),
    "readout_mode": str(READOUT_MODE),
    "attention_hidden_dim": int(ATTENTION_HIDDEN_DIM),
    "fc_hidden_dims": [int(x) for x in FC_HIDDEN_DIMS],
    "fc_inter_layer_dropout": float(FC_INTER_LAYER_DROPOUT),
    "fc_output_dropout": float(FC_OUTPUT_DROPOUT),
    "output_dim": int(len(TARGET_COLS)),
}

preprocess_config = {
    "csv_path": str(CSV_PATH),
    "seq_col": SEQ_COL,
    "group_col": GROUP_COL,
    "target_cols": list(TARGET_COLS),
    "vocab": dict(VOCAB),
    "pad_token": PAD_TOKEN,
    "pad_id": int(PAD_ID),
    "n_id": int(N_ID),
    "max_len": int(MAX_LEN),
    "target_mean": target_mean.tolist(),
    "target_std": target_std.tolist(),
    "split_summary": split_summary,
}

training_config = {
    "experiment_name": EXPERIMENT_NAME,
    "experiment_tag": EXPERIMENT_TAG,
    "run_name": RUN_NAME,
    "gpu_index": int(GPU_INDEX),
    "device": get_device_summary(DEVICE),
    "batch_size": int(BATCH_SIZE),
    "num_workers": int(NUM_WORKERS),
    "pin_memory": bool(PIN_MEMORY),
    "persistent_workers": bool(PERSISTENT_WORKERS),
    "seed": int(SEED),
    "optimizer_name": OPTIMIZER_NAME,
    "lr": float(LR),
    "weight_decay": float(WEIGHT_DECAY),
    "use_ema": bool(USE_EMA),
    "ema_decay": float(EMA_DECAY),
    "ema_update_after_step": int(EMA_UPDATE_AFTER_STEP),
    "ema_update_every": int(EMA_UPDATE_EVERY),
    "ema_eval_with_ema_weights": bool(EMA_EVAL_WITH_EMA_WEIGHTS),
    "num_epochs": int(NUM_EPOCHS),
    "use_warmup": bool(USE_WARMUP),
    "warmup_steps": int(WARMUP_STEPS),
    "warmup_start_factor": float(WARMUP_START_FACTOR),
    "use_scheduler": bool(USE_SCHEDULER),
    "scheduler_factor": float(SCHEDULER_FACTOR),
    "scheduler_patience": int(SCHEDULER_PATIENCE),
    "eval_every_n_steps": int(EVAL_EVERY_N_STEPS),
    "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
    "early_stopping_min_delta": float(EARLY_STOPPING_MIN_DELTA),
    "grad_clip_norm": float(GRAD_CLIP_NORM),
    "max_train_steps": int(MAX_TRAIN_STEPS),
}

display(pd.DataFrame([{
    "run_name": RUN_NAME,
    "model_class": model_config["model_class"],
    "use_cnn_before_lstm": model_config["use_cnn_before_lstm"],
    "cnn_layers": len(model_config["cnn_out_channels"]),
    "lstm_layers": len(model_config["lstm_hidden_dims"]),
    "fc_layers": len(model_config["fc_hidden_dims"]),
    "readout_mode": model_config["readout_mode"],
    "use_positional_embedding": model_config["use_positional_embedding"],
    "use_packed_lstm": model_config["use_packed_lstm"],
    "use_ema": training_config["use_ema"],
    "ema_decay": training_config["ema_decay"] if training_config["use_ema"] else None,
    "ema_update_after_step": training_config["ema_update_after_step"] if training_config["use_ema"] else None,
    "ema_update_every": training_config["ema_update_every"] if training_config["use_ema"] else None,
    "ema_eval_with_ema_weights": training_config["ema_eval_with_ema_weights"] if training_config["use_ema"] else None,
    "use_warmup": training_config["use_warmup"],
    "eval_every_n_steps": training_config["eval_every_n_steps"],
}]))


In [ ]:

class TALEDataset(Dataset):
    def __init__(self, df, seq_col, target_cols, max_len, target_mean=None, target_std=None):
        self.df = df.reset_index(drop=True)
        self.seq_col = seq_col
        self.target_cols = target_cols
        self.max_len = max_len
        self.target_mean = target_mean
        self.target_std = target_std

        self.seqs = self.df[self.seq_col].astype(str).tolist()
        self.y = self.df[self.target_cols].values.astype(np.float32, copy=False)

        encoded_items = [encode_sequence(seq, max_len=self.max_len) for seq in self.seqs]
        self.x_encoded = np.ascontiguousarray([x[0] for x in encoded_items], dtype=np.int64)
        self.seq_lens = np.ascontiguousarray([x[1] for x in encoded_items], dtype=np.int64)

        if self.target_mean is not None and self.target_std is not None:
            y_scaled = (self.y - self.target_mean) / self.target_std
        else:
            y_scaled = self.y

        self.y_scaled = np.ascontiguousarray(y_scaled, dtype=np.float32)

        # Configuration note.
        self.x_encoded = torch.from_numpy(self.x_encoded)
        self.seq_lens = torch.from_numpy(self.seq_lens)
        self.y_scaled = torch.from_numpy(self.y_scaled)
        self.n_samples = int(self.x_encoded.shape[0])

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        return self.x_encoded[idx], self.y_scaled[idx], self.seq_lens[idx]


In [ ]:

def build_datasets(seq_col, target_cols, max_len, target_mean, target_std):
    train_dataset = TALEDataset(
        train_df, seq_col, target_cols, max_len,
        target_mean=target_mean, target_std=target_std
    )
    val_dataset = TALEDataset(
        val_df, seq_col, target_cols, max_len,
        target_mean=target_mean, target_std=target_std
    )
    test_dataset = TALEDataset(
        test_df, seq_col, target_cols, max_len,
        target_mean=target_mean, target_std=target_std
    )
    return train_dataset, val_dataset, test_dataset


def build_dataloaders(train_dataset, val_dataset, test_dataset, batch_size, num_workers, pin_memory, persistent_workers):
    loader_kwargs = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    if num_workers > 0:
        loader_kwargs["persistent_workers"] = persistent_workers

    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
    return train_loader, val_loader, test_loader


train_dataset, val_dataset, test_dataset = build_datasets(
    SEQ_COL, TARGET_COLS, MAX_LEN, target_mean, target_std
)

train_loader, val_loader, test_loader = build_dataloaders(
    train_dataset, val_dataset, test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS
)

print("Train batches per epoch:", len(train_loader))
print("Val batches per epoch:  ", len(val_loader))
print("Test batches per epoch: ", len(test_loader))


independent_a549_dataset = None
independent_a549_loader = None
if EVALUATE_INDEPENDENT_A549 and independent_a549_df is not None:
    independent_target_indices = [TARGET_COLS.index(col) for col in INDEPENDENT_A549_MODEL_TARGET_COLS]
    independent_target_mean = target_mean[independent_target_indices]
    independent_target_std = target_std[independent_target_indices]

    independent_a549_dataset = TALEDataset(
        independent_a549_df,
        SEQ_COL,
        INDEPENDENT_A549_TARGET_COLS,
        MAX_LEN,
        target_mean=independent_target_mean,
        target_std=independent_target_std,
    )

    loader_kwargs = dict(
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS

    independent_a549_loader = DataLoader(independent_a549_dataset, shuffle=False, **loader_kwargs)
    print("Independent A549 batches:", len(independent_a549_loader))


In [ ]:
# =========================
# Configuration note.
# =========================
class AttentionReadout(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.0):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, mask=None, return_attention=False):
        attn_logits = self.score(x).squeeze(-1)  # [B, L]

        if mask is not None:
            attn_logits = attn_logits.masked_fill(~mask, -1e9)

        attn_weights = torch.softmax(attn_logits, dim=-1)  # [B, L]
        pooled = torch.sum(x * attn_weights.unsqueeze(-1), dim=1)  # [B, H]

        if return_attention:
            return pooled, attn_weights
        return pooled


class ConvBlock1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, pool_type="none", pool_size=1, activation="relu"):
        super().__init__()
        self.kernel_size = int(kernel_size)
        self.stride = int(stride)
        self.pool_type = str(pool_type).lower()
        self.pool_size = int(pool_size)
        self.activation = str(activation).lower()

        if self.activation != "relu":
            raise ValueError("Only relu activation is supported.")

        conv_padding = self.kernel_size // 2
        self.conv = nn.Conv1d(
            in_channels=int(in_channels),
            out_channels=int(out_channels),
            kernel_size=self.kernel_size,
            stride=self.stride,
            padding=conv_padding
        )
        self.act = nn.ReLU()

        if self.pool_type == "none":
            self.pool = nn.Identity()
        elif self.pool_type == "max":
            self.pool = nn.MaxPool1d(kernel_size=self.pool_size, stride=self.pool_size)
        else:
            raise ValueError("pool_type must be 'none' or 'max'.")

    def _conv_out_len(self, seq_lens):
        seq_lens = torch.as_tensor(seq_lens)
        padding = self.kernel_size // 2
        return torch.div(seq_lens + 2 * padding - self.kernel_size, self.stride, rounding_mode="floor") + 1

    def _pool_out_len(self, seq_lens):
        if self.pool_type == "none":
            return seq_lens
        return torch.div(seq_lens - self.pool_size, self.pool_size, rounding_mode="floor") + 1

    def output_lengths(self, seq_lens):
        seq_lens = self._conv_out_len(seq_lens)
        seq_lens = torch.clamp(seq_lens, min=1)
        seq_lens = self._pool_out_len(seq_lens)
        seq_lens = torch.clamp(seq_lens, min=1)
        return seq_lens

    def output_length_scalar(self, seq_len):
        seq_len = int(seq_len)
        padding = self.kernel_size // 2
        seq_len = (seq_len + 2 * padding - self.kernel_size) // self.stride + 1
        seq_len = max(seq_len, 1)
        if self.pool_type == "max":
            seq_len = (seq_len - self.pool_size) // self.pool_size + 1
            seq_len = max(seq_len, 1)
        return seq_len

    def forward(self, x, seq_lens=None):
        x = self.conv(x)
        x = self.act(x)
        x = self.pool(x)

        if seq_lens is not None:
            seq_lens = self.output_lengths(seq_lens).to(x.device)
            mask = torch.arange(x.size(-1), device=x.device).unsqueeze(0) < seq_lens.unsqueeze(1)
            x = x * mask.unsqueeze(1).to(x.dtype)

        return x, seq_lens


class TALELSTMRegressor(nn.Module):
    def __init__(
        self,
        vocab_size=6,
        pad_id=5,
        embed_dim=5,
        max_len=46,
        use_positional_embedding=True,
        positional_dropout=0.0,
        use_cnn_before_lstm=False,
        cnn_out_channels=None,
        cnn_kernel_sizes=None,
        cnn_strides=None,
        cnn_pool_types=None,
        cnn_pool_sizes=None,
        cnn_activation="relu",
        cnn_dropout=0.0,
        lstm_hidden_dims=None,
        lstm_bidirectional=None,
        lstm_inter_layer_dropout=0.0,
        lstm_output_dropout=0.0,
        use_packed_lstm=True,
        readout_mode="flatten",
        attention_hidden_dim=64,
        fc_hidden_dims=None,
        fc_inter_layer_dropout=0.0,
        fc_output_dropout=0.0,
        output_dim=4
    ):
        super().__init__()
        self.max_len = int(max_len)
        self.pad_id = int(pad_id)
        self.use_positional_embedding = bool(use_positional_embedding)
        self.use_cnn_before_lstm = bool(use_cnn_before_lstm)
        self.use_packed_lstm = bool(use_packed_lstm)
        self.readout_mode = str(readout_mode).lower()

        lstm_hidden_dims = list(lstm_hidden_dims or [])
        lstm_bidirectional = list(lstm_bidirectional or [])
        fc_hidden_dims = list(fc_hidden_dims or [])

        cnn_out_channels = list(cnn_out_channels or [])
        cnn_kernel_sizes = list(cnn_kernel_sizes or [])
        cnn_strides = list(cnn_strides or [])
        cnn_pool_types = list(cnn_pool_types or [])
        cnn_pool_sizes = list(cnn_pool_sizes or [])

        if len(lstm_hidden_dims) == 0:
            raise ValueError("Invalid value.")
        if len(lstm_hidden_dims) != len(lstm_bidirectional):
            raise ValueError("Invalid value.")
        if self.readout_mode not in {"flatten", "attention"}:
            raise ValueError("readout_mode must be 'flatten' or 'attention'.")

        if self.use_cnn_before_lstm:
            cnn_lengths = [len(cnn_out_channels), len(cnn_kernel_sizes), len(cnn_strides), len(cnn_pool_types), len(cnn_pool_sizes)]
            if len(set(cnn_lengths)) != 1 or len(cnn_out_channels) == 0:
                raise ValueError("Invalid value.")

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=self.pad_id
        )

        if self.use_positional_embedding:
            self.positional_embedding = nn.Embedding(max_len, embed_dim)
            self.positional_dropout = nn.Dropout(positional_dropout)
        else:
            self.positional_embedding = None
            self.positional_dropout = nn.Identity()

        # -------------------------
        # Configuration note.
        # -------------------------
        self.cnn_blocks = nn.ModuleList()
        self.cnn_dropout = nn.Dropout(float(cnn_dropout))
        current_input_size = embed_dim
        self.sequence_output_len = self.max_len

        if self.use_cnn_before_lstm:
            for out_channels, kernel_size, stride, pool_type, pool_size in zip(
                cnn_out_channels, cnn_kernel_sizes, cnn_strides, cnn_pool_types, cnn_pool_sizes
            ):
                block = ConvBlock1d(
                    in_channels=current_input_size,
                    out_channels=int(out_channels),
                    kernel_size=int(kernel_size),
                    stride=int(stride),
                    pool_type=str(pool_type),
                    pool_size=int(pool_size),
                    activation=str(cnn_activation)
                )
                self.cnn_blocks.append(block)
                current_input_size = int(out_channels)
                self.sequence_output_len = block.output_length_scalar(self.sequence_output_len)

        # -------------------------
        # Configuration note.
        # -------------------------
        self.lstm_layers = nn.ModuleList()
        self.lstm_inter_dropouts = nn.ModuleList()
        self.lstm_output_dim = None

        for layer_idx, (hidden_size, is_bidirectional) in enumerate(zip(lstm_hidden_dims, lstm_bidirectional)):
            lstm_layer = nn.LSTM(
                input_size=current_input_size,
                hidden_size=int(hidden_size),
                batch_first=True,
                bidirectional=bool(is_bidirectional)
            )
            self.lstm_layers.append(lstm_layer)
            current_input_size = int(hidden_size) * (2 if bool(is_bidirectional) else 1)
            self.lstm_output_dim = current_input_size

            if layer_idx < len(lstm_hidden_dims) - 1:
                self.lstm_inter_dropouts.append(nn.Dropout(float(lstm_inter_layer_dropout)))
            else:
                self.lstm_inter_dropouts.append(nn.Identity())

        self.lstm_output_dropout = nn.Dropout(float(lstm_output_dropout))

        # -------------------------
        # Configuration note.
        # -------------------------
        if self.readout_mode == "flatten":
            current_fc_in = self.sequence_output_len * self.lstm_output_dim
            self.readout = None
        else:
            self.readout = AttentionReadout(
                input_dim=self.lstm_output_dim,
                hidden_dim=attention_hidden_dim,
                dropout=float(fc_inter_layer_dropout)
            )
            current_fc_in = self.lstm_output_dim

        fc_layers = []
        for layer_idx, hidden_dim in enumerate(fc_hidden_dims):
            fc_layers.append(nn.Linear(current_fc_in, int(hidden_dim)))
            fc_layers.append(nn.ReLU())
            if layer_idx < len(fc_hidden_dims) - 1 and float(fc_inter_layer_dropout) > 0:
                fc_layers.append(nn.Dropout(float(fc_inter_layer_dropout)))
            current_fc_in = int(hidden_dim)

        self.fc_stack = nn.Sequential(*fc_layers) if len(fc_layers) > 0 else nn.Identity()
        self.fc_output_dropout = nn.Dropout(float(fc_output_dropout))
        self.fc_out = nn.Linear(current_fc_in, output_dim)

    def _run_cnn_frontend(self, x, seq_lens):
        if not self.use_cnn_before_lstm:
            return x, seq_lens

        mask = torch.arange(x.size(1), device=x.device).unsqueeze(0) < seq_lens.unsqueeze(1)
        x = x * mask.unsqueeze(-1).to(x.dtype)

        x = x.transpose(1, 2)  # [B, E, L]
        for block in self.cnn_blocks:
            x, seq_lens = block(x, seq_lens=seq_lens)
            x = self.cnn_dropout(x)
        x = x.transpose(1, 2)  # [B, L, C]
        return x, seq_lens

    def _run_single_lstm(self, lstm_layer, x, seq_lens):
        if self.use_packed_lstm:
            packed = nn.utils.rnn.pack_padded_sequence(
                x,
                lengths=seq_lens.detach().cpu(),
                batch_first=True,
                enforce_sorted=False
            )
            packed_out, _ = lstm_layer(packed)
            x, _ = nn.utils.rnn.pad_packed_sequence(
                packed_out,
                batch_first=True,
                total_length=self.sequence_output_len if self.use_cnn_before_lstm else self.max_len
            )
        else:
            x, _ = lstm_layer(x)
        return x

    def _run_stacked_lstm(self, x, seq_lens):
        for layer_idx, lstm_layer in enumerate(self.lstm_layers):
            x = self._run_single_lstm(lstm_layer, x, seq_lens)
            x = self.lstm_inter_dropouts[layer_idx](x)
        return x

    def forward(self, x, seq_lens=None, return_attention=False):
        if seq_lens is None:
            seq_lens = (x != self.pad_id).sum(dim=1)

        seq_lens = seq_lens.to(x.device)

        x = self.embedding(x)  # [B, L, E]

        if self.positional_embedding is not None:
            pos_ids = torch.arange(self.max_len, device=x.device).unsqueeze(0).expand(x.size(0), -1)
            x = x + self.positional_embedding(pos_ids)
            x = self.positional_dropout(x)

        x, seq_lens = self._run_cnn_frontend(x, seq_lens)
        x = self._run_stacked_lstm(x, seq_lens)
        x = self.lstm_output_dropout(x)

        if self.readout_mode == "flatten":
            x = x.contiguous().view(x.size(0), -1)
            attn_weights = None
        else:
            mask = torch.arange(x.size(1), device=x.device).unsqueeze(0) < seq_lens.unsqueeze(1)
            x, attn_weights = self.readout(x, mask=mask, return_attention=True)

        x = self.fc_stack(x)
        x = self.fc_output_dropout(x)
        x = self.fc_out(x)

        if return_attention:
            return x, attn_weights
        return x


def build_model_from_config(cfg):
    return TALELSTMRegressor(
        vocab_size=cfg["vocab_size"],
        pad_id=cfg.get("pad_id", PAD_ID),
        embed_dim=cfg["embed_dim"],
        max_len=cfg["max_len"],
        use_positional_embedding=cfg.get("use_positional_embedding", False),
        positional_dropout=cfg.get("positional_dropout", 0.0),
        use_cnn_before_lstm=cfg.get("use_cnn_before_lstm", False),
        cnn_out_channels=cfg.get("cnn_out_channels", []),
        cnn_kernel_sizes=cfg.get("cnn_kernel_sizes", []),
        cnn_strides=cfg.get("cnn_strides", []),
        cnn_pool_types=cfg.get("cnn_pool_types", []),
        cnn_pool_sizes=cfg.get("cnn_pool_sizes", []),
        cnn_activation=cfg.get("cnn_activation", "relu"),
        cnn_dropout=cfg.get("cnn_dropout", 0.0),
        lstm_hidden_dims=cfg["lstm_hidden_dims"],
        lstm_bidirectional=cfg["lstm_bidirectional"],
        lstm_inter_layer_dropout=cfg.get("lstm_inter_layer_dropout", 0.0),
        lstm_output_dropout=cfg.get("lstm_output_dropout", 0.0),
        use_packed_lstm=cfg.get("use_packed_lstm", True),
        readout_mode=cfg.get("readout_mode", "flatten"),
        attention_hidden_dim=cfg.get("attention_hidden_dim", 64),
        fc_hidden_dims=cfg.get("fc_hidden_dims", []),
        fc_inter_layer_dropout=cfg.get("fc_inter_layer_dropout", 0.0),
        fc_output_dropout=cfg.get("fc_output_dropout", 0.0),
        output_dim=cfg["output_dim"],
    )


model = build_model_from_config(model_config).to(DEVICE)

print(model)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {n_params:,}")
print(f"Trainable params: {n_trainable:,}")
if DEVICE.type == "cuda":
    print(f"Active GPU: {torch.cuda.current_device()} | {torch.cuda.get_device_name(torch.cuda.current_device())}")


In [ ]:

criterion = nn.MSELoss()

def clone_state_dict(state_dict, to_cpu=True):
    cloned = {}
    for k, v in state_dict.items():
        tensor = v.detach()
        if to_cpu:
            tensor = tensor.cpu()
        cloned[k] = tensor.clone()
    return cloned


def build_runtime_checkpoint(
    *,
    epoch,
    step_in_epoch,
    global_step,
    eval_index,
    best_val_loss,
    selected_weight_set,
    use_ema,
    ema_ready,
    model_state_dict,
    ema_model_state_dict,
    model_config,
    preprocess_config,
    training_config,
):
    # Configuration note.
    return {
        "epoch": int(epoch),
        "step_in_epoch": int(step_in_epoch),
        "global_step": int(global_step),
        "eval_index": int(eval_index),
        "best_val_loss": float(best_val_loss),
        "selected_weight_set": str(selected_weight_set),
        "ema_enabled": bool(use_ema),
        "ema_ready_at_save": bool(ema_ready),
        "model_state_dict": model_state_dict,
        "ema_model_state_dict": ema_model_state_dict,
        "model_config": to_builtin(model_config),
        "preprocess_config": to_builtin(preprocess_config),
        "training_config": to_builtin(training_config),
    }


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = float(decay)
        self.num_updates = 0
        self.shadow_params = {}
        self.shadow_buffers = {}

        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow_params[name] = param.detach().clone()

        for name, buf in model.named_buffers():
            self.shadow_buffers[name] = buf.detach().clone()

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        decay = float(self.decay)

        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            self.shadow_params[name].mul_(decay).add_(param.detach(), alpha=1.0 - decay)

        # Configuration note.
        for name, buf in model.named_buffers():
            self.shadow_buffers[name].copy_(buf.detach())

    def is_ready(self):
        return self.num_updates > 0

    @torch.no_grad()
    def store(self, model):
        stored_params = {}
        stored_buffers = {}

        for name, param in model.named_parameters():
            if param.requires_grad:
                stored_params[name] = param.detach().clone()

        for name, buf in model.named_buffers():
            stored_buffers[name] = buf.detach().clone()

        return stored_params, stored_buffers

    @torch.no_grad()
    def copy_to(self, model):
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            param.data.copy_(self.shadow_params[name].to(param.device))

        for name, buf in model.named_buffers():
            if name in self.shadow_buffers:
                buf.data.copy_(self.shadow_buffers[name].to(buf.device))

    @torch.no_grad()
    def restore(self, model, stored_params, stored_buffers):
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            param.data.copy_(stored_params[name].to(param.device))

        for name, buf in model.named_buffers():
            if name in stored_buffers:
                buf.data.copy_(stored_buffers[name].to(buf.device))

    @contextmanager
    def average_parameters(self, model):
        stored_params, stored_buffers = self.store(model)
        self.copy_to(model)
        try:
            yield model
        finally:
            self.restore(model, stored_params, stored_buffers)


def build_optimizer_and_scheduler(model, optimizer_name, lr, weight_decay, use_scheduler, scheduler_factor, scheduler_patience):
    if optimizer_name.lower() == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=float(lr),
            weight_decay=float(weight_decay)
        )
    elif optimizer_name.lower() == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=float(lr),
            weight_decay=float(weight_decay)
        )
    else:
        raise ValueError(f"Unsupported OPTIMIZER_NAME: {optimizer_name}")

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=float(scheduler_factor),
            patience=int(scheduler_patience)
        )
    return optimizer, scheduler


def set_lr(optimizer, lr_value):
    for param_group in optimizer.param_groups:
        param_group["lr"] = float(lr_value)


def get_warmup_lr(base_lr, global_step, warmup_steps, warmup_start_factor=0.1):
    if warmup_steps <= 0:
        return float(base_lr)
    step = min(max(int(global_step), 1), int(warmup_steps))
    start_lr = float(base_lr) * float(warmup_start_factor)
    progress = step / float(warmup_steps)
    return start_lr + (float(base_lr) - start_lr) * progress


optimizer, scheduler = build_optimizer_and_scheduler(
    model=model,
    optimizer_name=OPTIMIZER_NAME,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    use_scheduler=USE_SCHEDULER,
    scheduler_factor=SCHEDULER_FACTOR,
    scheduler_patience=SCHEDULER_PATIENCE,
)

ema_tracker = ModelEMA(model, decay=EMA_DECAY) if USE_EMA else None

print("Optimizer:", optimizer.__class__.__name__)
print("Initial LR:", optimizer.param_groups[0]["lr"])
print("Scheduler:", None if scheduler is None else scheduler.__class__.__name__)
print("EMA enabled:", USE_EMA)
if USE_EMA:
    print(
        "EMA settings:",
        {
            "decay": EMA_DECAY,
            "update_after_step": EMA_UPDATE_AFTER_STEP,
            "update_every": EMA_UPDATE_EVERY,
            "eval_with_ema_weights": EMA_EVAL_WITH_EMA_WEIGHTS,
        }
    )
print("Warmup enabled:", USE_WARMUP, "| warmup_steps:", WARMUP_STEPS, "| start_factor:", WARMUP_START_FACTOR)


In [ ]:

# =========================
# Evaluation functions
# =========================
def inverse_transform_targets(y_scaled, mean, std):
    return y_scaled * std + mean


def safe_pearsonr_fast(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    x = x - x.mean()
    y = y - y.mean()

    denom = np.sqrt(np.sum(x * x) * np.sum(y * y))
    if denom < 1e-12:
        return np.nan
    return float(np.sum(x * y) / denom)


def compute_metrics(y_true, y_pred, target_cols):
    metrics = {}

    diff = y_true - y_pred
    metrics["overall_rmse"] = float(np.sqrt(np.mean(diff * diff)))
    rmse_by_target = np.sqrt(np.mean(diff * diff, axis=0))

    for i, col in enumerate(target_cols):
        yt = y_true[:, i]
        yp = y_pred[:, i]

        metrics[f"{col}_rmse"] = float(rmse_by_target[i])
        metrics[f"{col}_pearson"] = safe_pearsonr_fast(yt, yp)

    return metrics


@torch.inference_mode()
def evaluate_loss_only(model, data_loader, criterion, device):
    """
    
    
    """
    model.eval()

    total_loss = 0.0
    n_samples = 0

    for xb, yb, seq_lens in data_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        seq_lens = seq_lens.to(device, non_blocking=True)

        pred = model(xb, seq_lens=seq_lens)
        loss = criterion(pred, yb)

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        n_samples += batch_size

    avg_loss = total_loss / max(n_samples, 1)
    return {"loss": float(avg_loss)}


@torch.inference_mode()
def evaluate_model(model, data_loader, criterion, device, target_mean, target_std, target_cols):
    model.eval()

    total_loss = 0.0
    n_samples = 0

    y_true_scaled_all = []
    y_pred_scaled_all = []

    for xb, yb, seq_lens in data_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        seq_lens = seq_lens.to(device, non_blocking=True)

        pred = model(xb, seq_lens=seq_lens)
        loss = criterion(pred, yb)

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        n_samples += batch_size

        y_true_scaled_all.append(yb.cpu())
        y_pred_scaled_all.append(pred.cpu())

    avg_loss = total_loss / max(n_samples, 1)

    y_true_scaled = torch.cat(y_true_scaled_all, dim=0).numpy()
    y_pred_scaled = torch.cat(y_pred_scaled_all, dim=0).numpy()

    y_true = inverse_transform_targets(y_true_scaled, target_mean, target_std)
    y_pred = inverse_transform_targets(y_pred_scaled, target_mean, target_std)

    metrics = compute_metrics(y_true, y_pred, target_cols)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


@torch.inference_mode()
def evaluate_model_output_subset(
    model,
    data_loader,
    criterion,
    device,
    target_mean,
    target_std,
    target_cols,
    output_indices,
):
    model.eval()

    total_loss = 0.0
    n_samples = 0

    y_true_scaled_all = []
    y_pred_scaled_all = []

    output_indices = list(output_indices)

    for xb, yb, seq_lens in data_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        seq_lens = seq_lens.to(device, non_blocking=True)

        pred_full = model(xb, seq_lens=seq_lens)
        pred = pred_full[:, output_indices]
        loss = criterion(pred, yb)

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        n_samples += batch_size

        y_true_scaled_all.append(yb.cpu())
        y_pred_scaled_all.append(pred.cpu())

    avg_loss = total_loss / max(n_samples, 1)

    y_true_scaled = torch.cat(y_true_scaled_all, dim=0).numpy()
    y_pred_scaled = torch.cat(y_pred_scaled_all, dim=0).numpy()

    y_true = inverse_transform_targets(y_true_scaled, target_mean, target_std)
    y_pred = inverse_transform_targets(y_pred_scaled, target_mean, target_std)

    metrics = compute_metrics(y_true, y_pred, target_cols)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


In [ ]:

# =========================
# Training loop
# =========================
def run_training(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    num_epochs,
    target_mean,
    target_std,
    target_cols,
    grad_clip_norm,
    max_train_steps,
    eval_every_n_steps,
    use_warmup,
    warmup_steps,
    warmup_start_factor,
    ema_tracker=None,
    use_ema=False,
    ema_update_after_step=0,
    ema_update_every=1,
    ema_eval_with_ema_weights=True,
):
    use_ema = bool(use_ema and ema_tracker is not None)

    history_epoch = {
        "epoch": [],
        "global_step_end": [],
        "train_loss_epoch": [],
        "n_steps_epoch": [],
        "last_eval_global_step": [],
        "last_eval_val_loss": [],
        "last_eval_weight_set": [],
        "lr_epoch_end": [],
    }
    history_step_eval = {
        "eval_index": [],
        "epoch": [],
        "step_in_epoch": [],
        "global_step": [],
        "train_loss_since_last_eval": [],
        "val_loss": [],
        "eval_weight_set": [],
        "lr": [],
    }

    best_val_loss = float("inf")
    best_epoch = -1
    best_global_step = -1
    best_step_in_epoch = -1
    best_checkpoint_path = None
    patience_counter = 0
    global_step = 0
    eval_index = 0
    stop_training = False

    since_eval_loss_sum = 0.0
    since_eval_n_samples = 0
    last_eval_global_step = 0
    last_eval_val_loss = np.nan
    last_eval_weight_set = "raw"

    def maybe_run_eval(epoch, step_in_epoch, force_eval=False):
        nonlocal eval_index, patience_counter, best_val_loss, best_epoch, best_global_step, best_step_in_epoch
        nonlocal since_eval_loss_sum, since_eval_n_samples, last_eval_global_step, last_eval_val_loss, last_eval_weight_set
        nonlocal stop_training, best_checkpoint_path

        should_eval = force_eval or (global_step % int(eval_every_n_steps) == 0)
        if not should_eval:
            return None
        if since_eval_n_samples <= 0:
            return None

        train_loss_since_last_eval = since_eval_loss_sum / max(since_eval_n_samples, 1)

        ema_ready = bool(use_ema and ema_tracker.is_ready())
        eval_weight_set = "ema" if (ema_ready and ema_eval_with_ema_weights) else "raw"

        if eval_weight_set == "ema":
            with ema_tracker.average_parameters(model):
                val_loss_dict = evaluate_loss_only(model, val_loader, criterion, device)
        else:
            val_loss_dict = evaluate_loss_only(model, val_loader, criterion, device)

        val_loss = float(val_loss_dict["loss"])

        if scheduler is not None:
            scheduler.step(val_loss)

        current_lr = float(optimizer.param_groups[0]["lr"])
        eval_index += 1

        history_step_eval["eval_index"].append(int(eval_index))
        history_step_eval["epoch"].append(int(epoch))
        history_step_eval["step_in_epoch"].append(int(step_in_epoch))
        history_step_eval["global_step"].append(int(global_step))
        history_step_eval["train_loss_since_last_eval"].append(float(train_loss_since_last_eval))
        history_step_eval["val_loss"].append(val_loss)
        history_step_eval["eval_weight_set"].append(str(eval_weight_set))
        history_step_eval["lr"].append(current_lr)

        print(
            f"Eval {eval_index:04d} | epoch={epoch:03d} | step_in_epoch={step_in_epoch:05d} | "
            f"global_step={global_step:07d} | train_loss={train_loss_since_last_eval:.6f} | "
            f"val_loss({eval_weight_set})={val_loss:.6f} | lr={current_lr:.2e}"
        )

        improved = (best_val_loss - val_loss) > float(EARLY_STOPPING_MIN_DELTA)
        if improved:
            best_val_loss = val_loss
            best_epoch = int(epoch)
            best_global_step = int(global_step)
            best_step_in_epoch = int(step_in_epoch)
            patience_counter = 0

            raw_model_state_dict = clone_state_dict(model.state_dict(), to_cpu=True)
            ema_model_state_dict = None

            if ema_ready:
                with ema_tracker.average_parameters(model):
                    ema_model_state_dict = clone_state_dict(model.state_dict(), to_cpu=True)

            checkpoint = build_runtime_checkpoint(
                epoch=epoch,
                step_in_epoch=step_in_epoch,
                global_step=global_step,
                eval_index=eval_index,
                best_val_loss=best_val_loss,
                selected_weight_set=eval_weight_set,
                use_ema=use_ema,
                ema_ready=ema_ready,
                model_state_dict=raw_model_state_dict,
                ema_model_state_dict=ema_model_state_dict,
                model_config=model_config,
                preprocess_config=preprocess_config,
                training_config=training_config,
            )
            torch.save(checkpoint, BEST_CHECKPOINT_TMP_PATH)
            best_checkpoint_path = BEST_CHECKPOINT_TMP_PATH
            print(f"    -> saved best checkpoint to {BEST_CHECKPOINT_TMP_PATH}")
        else:
            patience_counter += 1

        since_eval_loss_sum = 0.0
        since_eval_n_samples = 0
        last_eval_global_step = int(global_step)
        last_eval_val_loss = float(val_loss)
        last_eval_weight_set = str(eval_weight_set)

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping triggered at global_step {global_step}. "
                f"Best epoch = {best_epoch}, best step = {best_global_step}"
            )
            stop_training = True

        model.train()
        return val_loss

    for epoch in range(1, int(num_epochs) + 1):
        model.train()

        running_loss = 0.0
        n_samples = 0
        step_in_epoch = 0

        for xb, yb, seq_lens in train_loader:
            xb = xb.to(device, non_blocking=PIN_MEMORY)
            yb = yb.to(device, non_blocking=PIN_MEMORY)
            seq_lens = seq_lens.to(device, non_blocking=PIN_MEMORY)

            optimizer.zero_grad(set_to_none=True)

            if use_warmup and warmup_steps > 0 and global_step < warmup_steps:
                warmup_lr = get_warmup_lr(
                    base_lr=LR,
                    global_step=global_step + 1,
                    warmup_steps=warmup_steps,
                    warmup_start_factor=warmup_start_factor
                )
                set_lr(optimizer, warmup_lr)

            pred = model(xb, seq_lens=seq_lens)
            loss = criterion(pred, yb)
            loss.backward()

            if grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(grad_clip_norm))

            optimizer.step()

            global_step += 1
            if use_ema:
                if global_step > int(ema_update_after_step):
                    steps_since_start = global_step - int(ema_update_after_step)
                    if steps_since_start % int(ema_update_every) == 0:
                        ema_tracker.update(model)

            batch_size_this = xb.size(0)
            running_loss += loss.item() * batch_size_this
            n_samples += batch_size_this
            since_eval_loss_sum += loss.item() * batch_size_this
            since_eval_n_samples += batch_size_this

            step_in_epoch += 1

            maybe_run_eval(epoch=epoch, step_in_epoch=step_in_epoch, force_eval=False)

            if max_train_steps > 0 and global_step >= max_train_steps:
                stop_training = True
                break
            if stop_training:
                break

        # Configuration note.
        if global_step > last_eval_global_step and since_eval_n_samples > 0:
            maybe_run_eval(epoch=epoch, step_in_epoch=step_in_epoch, force_eval=True)

        train_loss = running_loss / max(n_samples, 1)
        current_lr = float(optimizer.param_groups[0]["lr"])

        history_epoch["epoch"].append(int(epoch))
        history_epoch["global_step_end"].append(int(global_step))
        history_epoch["train_loss_epoch"].append(float(train_loss))
        history_epoch["n_steps_epoch"].append(int(step_in_epoch))
        history_epoch["last_eval_global_step"].append(int(last_eval_global_step))
        history_epoch["last_eval_val_loss"].append(float(last_eval_val_loss) if not np.isnan(last_eval_val_loss) else np.nan)
        history_epoch["last_eval_weight_set"].append(str(last_eval_weight_set))
        history_epoch["lr_epoch_end"].append(current_lr)

        print(
            f"Epoch {epoch:03d} done | global_step={global_step:07d} | "
            f"train_loss_epoch={train_loss:.6f} | last_eval_val_loss({last_eval_weight_set})={last_eval_val_loss:.6f} | "
            f"lr={current_lr:.2e}"
        )

        if stop_training:
            if max_train_steps > 0 and global_step >= max_train_steps:
                print(f"Reached MAX_TRAIN_STEPS={max_train_steps}; stopping after epoch {epoch}.")
            break

    return {
        "history_epoch": history_epoch,
        "history_step_eval": history_step_eval,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "best_global_step": best_global_step,
        "best_step_in_epoch": best_step_in_epoch,
        "best_checkpoint_path": best_checkpoint_path,
    }


train_result = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    target_mean=target_mean,
    target_std=target_std,
    target_cols=TARGET_COLS,
    grad_clip_norm=GRAD_CLIP_NORM,
    max_train_steps=MAX_TRAIN_STEPS,
    eval_every_n_steps=EVAL_EVERY_N_STEPS,
    use_warmup=USE_WARMUP,
    warmup_steps=WARMUP_STEPS,
    warmup_start_factor=WARMUP_START_FACTOR,
    ema_tracker=ema_tracker,
    use_ema=USE_EMA,
    ema_update_after_step=EMA_UPDATE_AFTER_STEP,
    ema_update_every=EMA_UPDATE_EVERY,
    ema_eval_with_ema_weights=EMA_EVAL_WITH_EMA_WEIGHTS,
)

history_epoch = train_result["history_epoch"]
history_step_eval = train_result["history_step_eval"]
best_val_loss = train_result["best_val_loss"]
best_epoch = train_result["best_epoch"]
best_global_step = train_result["best_global_step"]
best_step_in_epoch = train_result["best_step_in_epoch"]
best_checkpoint_path = train_result["best_checkpoint_path"]

history_epoch_df = pd.DataFrame(history_epoch)
history_step_eval_df = pd.DataFrame(history_step_eval)

display(history_step_eval_df.tail())
display(history_epoch_df.tail())

print("Best epoch:", best_epoch)
print("Best global step:", best_global_step)
print("Best step within epoch:", best_step_in_epoch)
print("Best val loss:", best_val_loss)

if SAVE_DEBUG_ARTIFACTS:
    history_step_eval_df.to_csv(STEP_EVAL_HISTORY_PATH, index=False)
    history_epoch_df.to_csv(EPOCH_HISTORY_PATH, index=False)
    print("Step-eval history saved to:", STEP_EVAL_HISTORY_PATH)
    print("Epoch history saved to:", EPOCH_HISTORY_PATH)

if best_checkpoint_path is None:
    raise RuntimeError("No best checkpoint was generated during training; check the training logic.")

shutil.copy2(best_checkpoint_path, FINAL_MODEL_SAVE_PATH)
print("Final model path:", FINAL_MODEL_SAVE_PATH)

if SAVE_DEBUG_ARTIFACTS and COPY_NOTEBOOK_SNAPSHOT:
    saved_snapshot = save_notebook_snapshot(CURRENT_NOTEBOOK_PATH, NOTEBOOK_SNAPSHOT_PATH)
    print("Notebook snapshot saved:" if saved_snapshot else "Notebook snapshot not found:", NOTEBOOK_SNAPSHOT_PATH)


In [ ]:

history_step_eval_df = pd.DataFrame(history_step_eval)
history_epoch_df = pd.DataFrame(history_epoch)

display(history_step_eval_df.tail())
display(history_epoch_df.tail())


In [ ]:
# Configuration note.
plot_df = history_step_eval_df.copy()

if len(plot_df) > 0:
    plt.figure(figsize=(7, 5))

    train_vals = plot_df["train_loss_since_last_eval"]
    if len(train_vals) > 1 and (train_vals.max() - train_vals.min()) > 0:
        train_norm = (train_vals - train_vals.min()) / (train_vals.max() - train_vals.min())
    else:
        train_norm = train_vals * 0.0

    val_vals = plot_df["val_loss"]
    if len(val_vals) > 1 and (val_vals.max() - val_vals.min()) > 0:
        val_norm = (val_vals - val_vals.min()) / (val_vals.max() - val_vals.min())
    else:
        val_norm = val_vals * 0.0

    plt.plot(plot_df["global_step"], train_norm, label="train_loss")
    plt.plot(plot_df["global_step"], val_norm, label="val_loss")

    epoch_line_added = False
    if len(history_epoch_df) > 0:
        steps_per_epoch = len(train_loader)

        y_top = max(
            float(np.nanmax(train_norm)) if len(train_norm) > 0 else 1.0,
            float(np.nanmax(val_norm)) if len(val_norm) > 0 else 1.0,
        )
        y_top = y_top if np.isfinite(y_top) else 1.0

        for _, row in history_epoch_df.iterrows():
            epoch_idx = int(row["epoch"])
            x = row["global_step_end"]

            expected_epoch_end = epoch_idx * steps_per_epoch

            # Configuration note.
            if x < expected_epoch_end:
                continue

            plt.axvline(
                x=expected_epoch_end,
                linestyle="--",
                linewidth=0.8,
                alpha=0.35,
                label="epoch end" if not epoch_line_added else None,
            )
            epoch_line_added = True

            plt.text(
                expected_epoch_end,
                y_top + 0.03,
                f"e{epoch_idx}",
                rotation=90,
                va="bottom",
                ha="center",
                fontsize=8,
                alpha=0.7,
            )

    plt.xlabel("Global step")
    plt.ylabel("Normalized loss")
    plt.title("Training curves (step-level eval, normalized)")
    plt.legend()
    plt.tight_layout()

    if SAVE_DEBUG_ARTIFACTS:
        plt.savefig(TRAIN_CURVE_PATH, dpi=200)

    plt.show()

    if SAVE_DEBUG_ARTIFACTS:
        print("Saved:", TRAIN_CURVE_PATH)
else:
    print("No step-level eval curve to plot.")

In [ ]:

# =========================
# Load best model
# =========================
checkpoint = torch.load(
    FINAL_MODEL_SAVE_PATH,
    map_location=DEVICE,
    weights_only=False
)

loaded_model_config = checkpoint["model_config"]

best_model = build_model_from_config(loaded_model_config).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

best_model_ema = None
if checkpoint.get("ema_model_state_dict") is not None:
    best_model_ema = build_model_from_config(loaded_model_config).to(DEVICE)
    best_model_ema.load_state_dict(checkpoint["ema_model_state_dict"])
    best_model_ema.eval()

selected_weight_set = str(checkpoint.get("selected_weight_set", "raw"))
if selected_weight_set == "ema" and best_model_ema is None:
    selected_weight_set = "raw"

print("Loaded best model from:", FINAL_MODEL_SAVE_PATH)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Checkpoint step_in_epoch:", checkpoint.get("step_in_epoch"))
print("Checkpoint global_step:", checkpoint.get("global_step"))
print("Checkpoint best val loss:", checkpoint["best_val_loss"])
print("Selected weight set:", selected_weight_set)
print("EMA weights available:", best_model_ema is not None)
display(pd.DataFrame([loaded_model_config]))


In [ ]:

# =========================
# Configuration note.
# =========================
def evaluate_all_splits(model_for_eval):
    train_metrics, train_true, train_pred = evaluate_model(
        model_for_eval, train_loader, criterion, DEVICE, target_mean, target_std, TARGET_COLS
    )
    val_metrics, val_true, val_pred = evaluate_model(
        model_for_eval, val_loader, criterion, DEVICE, target_mean, target_std, TARGET_COLS
    )
    test_metrics, test_true, test_pred = evaluate_model(
        model_for_eval, test_loader, criterion, DEVICE, target_mean, target_std, TARGET_COLS
    )

    independent_a549_metrics = None
    independent_a549_true = None
    independent_a549_pred = None
    if EVALUATE_INDEPENDENT_A549 and independent_a549_loader is not None:
        independent_a549_metrics, independent_a549_true, independent_a549_pred = evaluate_model_output_subset(
            model_for_eval,
            independent_a549_loader,
            criterion,
            DEVICE,
            target_mean=independent_target_mean,
            target_std=independent_target_std,
            target_cols=INDEPENDENT_A549_TARGET_COLS,
            output_indices=independent_target_indices,
        )

    return {
        "train_metrics": train_metrics,
        "train_true": train_true,
        "train_pred": train_pred,
        "val_metrics": val_metrics,
        "val_true": val_true,
        "val_pred": val_pred,
        "test_metrics": test_metrics,
        "test_true": test_true,
        "test_pred": test_pred,
        "independent_a549_metrics": independent_a549_metrics,
        "independent_a549_true": independent_a549_true,
        "independent_a549_pred": independent_a549_pred,
    }


eval_results_by_weight = {
    "raw": evaluate_all_splits(best_model)
}
if best_model_ema is not None:
    eval_results_by_weight["ema"] = evaluate_all_splits(best_model_ema)

if selected_weight_set not in eval_results_by_weight:
    selected_weight_set = "raw"

selected_eval_results = eval_results_by_weight[selected_weight_set]

train_metrics = selected_eval_results["train_metrics"]
train_true = selected_eval_results["train_true"]
train_pred = selected_eval_results["train_pred"]

val_metrics = selected_eval_results["val_metrics"]
val_true = selected_eval_results["val_true"]
val_pred = selected_eval_results["val_pred"]

test_metrics = selected_eval_results["test_metrics"]
test_true = selected_eval_results["test_true"]
test_pred = selected_eval_results["test_pred"]

independent_a549_metrics = selected_eval_results["independent_a549_metrics"]
independent_a549_true = selected_eval_results["independent_a549_true"]
independent_a549_pred = selected_eval_results["independent_a549_pred"]


def metrics_to_table(metrics, split_name, target_cols, weight_set):
    rows = []
    rows.append({
        "weight_set": weight_set,
        "split": split_name,
        "target": "OVERALL",
        "loss": metrics["loss"],
        "rmse": metrics["overall_rmse"],
        "pearson": np.nan
    })
    for col in target_cols:
        rows.append({
            "weight_set": weight_set,
            "split": split_name,
            "target": col,
            "loss": np.nan,
            "rmse": metrics[f"{col}_rmse"],
            "pearson": metrics[f"{col}_pearson"]
        })
    return pd.DataFrame(rows)


metrics_tables = []
for weight_set_name, result_dict in eval_results_by_weight.items():
    metrics_tables.extend([
        metrics_to_table(result_dict["train_metrics"], "train", TARGET_COLS, weight_set_name),
        metrics_to_table(result_dict["val_metrics"], "val", TARGET_COLS, weight_set_name),
        metrics_to_table(result_dict["test_metrics"], "test", TARGET_COLS, weight_set_name),
    ])
    if result_dict["independent_a549_metrics"] is not None:
        metrics_tables.append(
            metrics_to_table(
                result_dict["independent_a549_metrics"],
                "independent_a549",
                INDEPENDENT_A549_TARGET_COLS,
                weight_set_name
            )
        )

metrics_table = pd.concat(metrics_tables, ignore_index=True)

print("Selected evaluation weight set for downstream plots/exports:", selected_weight_set)
display(metrics_table)


In [ ]:
# Configuration note.
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes = axes.flatten()

for i, col in enumerate(TARGET_COLS):
    ax = axes[i]
    yt = test_true[:, i]
    yp = test_pred[:, i]

    ax.scatter(yt, yp, alpha=0.5, s=12)
    min_v = min(yt.min(), yp.min())
    max_v = max(yt.max(), yp.max())
    ax.plot([min_v, max_v], [min_v, max_v], linestyle="--")
    ax.set_title(
        f"{col}\nRMSE={test_metrics[f'{col}_rmse']:.4f}, "
        f"Pearson={test_metrics[f'{col}_pearson']:.3f}"
    )
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")

plt.tight_layout()
if SAVE_DEBUG_ARTIFACTS:
    plt.savefig(TEST_SCATTER_PATH, dpi=200)
plt.show()

if SAVE_DEBUG_ARTIFACTS:
    print("Saved:", TEST_SCATTER_PATH)

if independent_a549_metrics is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    axes = np.atleast_1d(axes)

    for i, col in enumerate(INDEPENDENT_A549_TARGET_COLS):
        ax = axes[i]
        yt = independent_a549_true[:, i]
        yp = independent_a549_pred[:, i]

        ax.scatter(yt, yp, alpha=0.5, s=12)
        min_v = min(yt.min(), yp.min())
        max_v = max(yt.max(), yp.max())
        ax.plot([min_v, max_v], [min_v, max_v], linestyle="--")
        ax.set_title(
            f"{col}\nRMSE={independent_a549_metrics[f'{col}_rmse']:.4f}, "
            f"Pearson={independent_a549_metrics[f'{col}_pearson']:.3f}"
        )
        ax.set_xlabel("True")
        ax.set_ylabel("Predicted")

    plt.tight_layout()
    if SAVE_DEBUG_ARTIFACTS:
        plt.savefig(INDEPENDENT_A549_SCATTER_PATH, dpi=200)
    plt.show()

    if SAVE_DEBUG_ARTIFACTS:
        print("Saved:", INDEPENDENT_A549_SCATTER_PATH)


In [ ]:

# =========================
# Configuration note.
# =========================
test_pred_df = test_df[[SEQ_COL, GROUP_COL] + TARGET_COLS].copy()

for i, col in enumerate(TARGET_COLS):
    test_pred_df[f"pred.{col}"] = test_pred[:, i]
    test_pred_df[f"resid.{col}"] = test_pred_df[f"pred.{col}"] - test_pred_df[col]

independent_a549_pred_df = None
if independent_a549_metrics is not None:
    independent_a549_pred_df = independent_a549_df[[SEQ_COL, "dna"] + INDEPENDENT_A549_TARGET_COLS].copy() if "dna" in independent_a549_df.columns else independent_a549_df[[SEQ_COL] + INDEPENDENT_A549_TARGET_COLS].copy()

    for i, col in enumerate(INDEPENDENT_A549_TARGET_COLS):
        independent_a549_pred_df[f"pred.{col}"] = independent_a549_pred[:, i]
        independent_a549_pred_df[f"resid.{col}"] = independent_a549_pred_df[f"pred.{col}"] - independent_a549_pred_df[col]

artifact_manifest = {
    "model_bundle_path": str(FINAL_MODEL_SAVE_PATH),
    "summary_path": str(SUMMARY_PATH) if SAVE_SUMMARY_JSON else None,
    "test_prediction_path": str(OUT_TEST_PRED) if SAVE_PREDICTIONS else None,
    "independent_a549_prediction_path": str(OUT_INDEPENDENT_A549_PRED) if (SAVE_PREDICTIONS and independent_a549_pred_df is not None) else None,
    "selected_weight_set": str(selected_weight_set),
    "debug_artifacts": {
        "enabled": bool(SAVE_DEBUG_ARTIFACTS),
        "step_eval_history_path": str(STEP_EVAL_HISTORY_PATH) if SAVE_DEBUG_ARTIFACTS else None,
        "epoch_history_path": str(EPOCH_HISTORY_PATH) if SAVE_DEBUG_ARTIFACTS else None,
        "train_curve_path": str(TRAIN_CURVE_PATH) if SAVE_DEBUG_ARTIFACTS else None,
        "test_scatter_path": str(TEST_SCATTER_PATH) if SAVE_DEBUG_ARTIFACTS else None,
        "independent_a549_scatter_path": str(INDEPENDENT_A549_SCATTER_PATH) if (SAVE_DEBUG_ARTIFACTS and independent_a549_metrics is not None) else None,
        "notebook_snapshot_path": str(NOTEBOOK_SNAPSHOT_PATH) if (SAVE_DEBUG_ARTIFACTS and COPY_NOTEBOOK_SNAPSHOT) else None,
    }
}

raw_eval_metrics = {
    "train_metrics": eval_results_by_weight["raw"]["train_metrics"],
    "val_metrics": eval_results_by_weight["raw"]["val_metrics"],
    "test_metrics": eval_results_by_weight["raw"]["test_metrics"],
    "independent_a549_metrics": eval_results_by_weight["raw"]["independent_a549_metrics"],
}

ema_eval_metrics = None
if "ema" in eval_results_by_weight:
    ema_eval_metrics = {
        "train_metrics": eval_results_by_weight["ema"]["train_metrics"],
        "val_metrics": eval_results_by_weight["ema"]["val_metrics"],
        "test_metrics": eval_results_by_weight["ema"]["test_metrics"],
        "independent_a549_metrics": eval_results_by_weight["ema"]["independent_a549_metrics"],
    }

selected_eval_metrics = {
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "independent_a549_metrics": independent_a549_metrics,
}

final_metrics = {
    "selected_weight_set": str(selected_weight_set),
    "selected_eval_metrics": selected_eval_metrics,
    "raw_eval_metrics": raw_eval_metrics,
    "ema_eval_metrics": ema_eval_metrics,
    "best_epoch": int(best_epoch),
    "best_global_step": int(best_global_step),
    "best_step_in_epoch": int(best_step_in_epoch),
    "best_val_loss": float(best_val_loss),
    "final_model_path": str(FINAL_MODEL_SAVE_PATH),
}

summary_payload = {
    "artifact_manifest": artifact_manifest,
    "final_metrics": final_metrics,
    "split_summary": split_summary,
    "target_stats": target_stats.to_dict(orient="records"),
    "model_config": to_builtin(model_config),
    "preprocess_config": to_builtin(preprocess_config),
    "training_config": to_builtin(training_config),
    "timestamp": datetime.now().isoformat(),
}

# Configuration note.
model_bundle = {
    "epoch": int(best_epoch),
    "step_in_epoch": int(best_step_in_epoch),
    "global_step": int(best_global_step),
    "best_val_loss": float(best_val_loss),
    "selected_weight_set": str(selected_weight_set),
    "model_state_dict": clone_state_dict(best_model.state_dict()),
    "ema_model_state_dict": None if best_model_ema is None else clone_state_dict(best_model_ema.state_dict()),
    "model_config": to_builtin(model_config),
    "preprocess_config": to_builtin(preprocess_config),
    "training_config": to_builtin(training_config),
    "summary": to_builtin(summary_payload),
}
torch.save(model_bundle, FINAL_MODEL_SAVE_PATH)
print("Updated final model bundle:", FINAL_MODEL_SAVE_PATH)

if SAVE_PREDICTIONS:
    test_pred_df.to_csv(OUT_TEST_PRED, index=False)
    print("Saved:", OUT_TEST_PRED)
    if independent_a549_pred_df is not None:
        independent_a549_pred_df.to_csv(OUT_INDEPENDENT_A549_PRED, index=False)
        print("Saved:", OUT_INDEPENDENT_A549_PRED)

if SAVE_SUMMARY_JSON:
    with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
        json.dump(to_builtin(summary_payload), f, ensure_ascii=False, indent=2)
    print("Saved:", SUMMARY_PATH)


## Suggested Follow-up Experiments

Recommended isolated comparisons include readout mode, position/padding controls, dropout placement, and EMA settings.